In [ ]:
from itertools import product

def generar_combinaciones(elem_1,elem_2,elem_3,elem_4)->list:
    """
    Crea combinaciones de 'r' elementos a partir de una lista.
    El orden no importa y no se repiten elementos.
    """
    # combinations(lista, tamaño) devuelve un iterador
    comb = list(product(elem_1,elem_2,elem_3,elem_4))
    
    # Convertimos el iterador a lista
    return list(comb)

In [ ]:
from supervivencia import Supervivencia
from mutacion import Mutacion
from cruze import Cruce_cargas
import pandas as pd

def guardar_configuracion(configuraciones_posibles:list[list[Cruce_cargas,Mutacion,str,Supervivencia]],nombre_archivo="datos")->None:
    ruta_acceso:str = "/home/oarroyo/computo_evolutivo_proyectos/datos_experimentos/archivos_configuraciones/"
    pf:pd.DataFrame = pd.DataFrame(configuraciones_posibles,columns=['Cruce','Mutacion','Penalizacion','Supervivencia'])
    pf.to_csv(ruta_acceso+nombre_archivo+".csv",index=True)
    

In [ ]:
from supervivencia import Con_elitismo, Sin_elitismo, Supervivencia
from penalizacion import Penalizacion, Lineal, Fitness_Cero, Cuadratica, Exponencial
from carga import Carga
from grafica import Grafica, Grafica_lineas, Grafica_lineas_dobles
from fitness import Fitness
import numpy as np
import numpy.typing as npt
from mutacion import Mutacion, Seleccion_aleatoria, Tasa_mutacion
from cruze import Cruce_cargas, Cruce_uniforme, Cruce_un_punto, Cruce_dos_punto
from contenedor import Contenedor
from diversidad import Diversidad


def evolucion(penalizacion_opcion: str = "fitnes_cero",
              tamano_poblacion: int = 100,
              max_generaciones: int = 100, 
              recombinacion_opcion:Cruce_cargas = Cruce_dos_punto,
              mutacion_opcion:Mutacion = Seleccion_aleatoria,
              supervivencia_opcion: Supervivencia = Con_elitismo,
              configuracion_opcion:str = "base"):
    
    # Métricas por generación
    valor_minimo_generacion: list[int] = []
    valor_maximo_generacion: list[int] = []
    valor_promedio_generacion: list[float] = []
    valor_desviacion_std_generacion: list[float] = []
    valor_varianza_generacion: list[float] = []
    valor_diversidad_generacion: list[float] = []

    # Inicialización
    carga = Carga()
    cargas: list[npt.NDArray[np.int32]] = carga.generar_carga()

    mejor_generacion_anterior: Contenedor = None

    supervivencia: Supervivencia = supervivencia_opcion()
    fitness: Fitness 
    diversidad: Diversidad = Diversidad()
    poblacion: list[Contenedor] = []

    # Evolución
    for generacion in range(max_generaciones):
        
        aptitudes:list[int] = []

        # Llevamos la carga a los contenedores
        for carga in cargas:
            contenedor = Contenedor(carga=carga)
            poblacion.append(contenedor)

        # Calculamos su fitnes
        for contenedor in poblacion:
            fitness = Fitness(carga=contenedor.carga)
            aptitudes.append(fitness.calcular_fitness(penalizacion_selecionada=penalizacion_opcion))
        
        # Llevamos su fitnes al contenedor    
        for posicion, fitness in enumerate(aptitudes):
            poblacion[posicion].fitness = fitness

        if not mejor_generacion_anterior == None:
            posicion, carga_minima = min(enumerate(poblacion),
                                            key=lambda x: x[1].fitness)
            
            poblacion[posicion] = mejor_generacion_anterior

        # Esto se debe sacar de los objetos
        aptitudes = []
        for contenedor in poblacion:
            aptitudes.append(contenedor.fitness)

        valor_maximo:int = (np.max(aptitudes))
        valor_minimo:int = (np.min(aptitudes))
        valor_promedio: float = (np.mean(aptitudes))
        valor_desviacion_std: float = (np.std(aptitudes))
        valor_varianza: float = (np.var(aptitudes))
        valor_divercidad:float = diversidad.calcular_diversidad(poblacion=poblacion)

        nueva_generacion,mejor_generacion_anterior = supervivencia.seleccion_supervivencia( poblacion=poblacion,
                                                                            tamano_poblacion=tamano_poblacion,
                                                                            generacion=generacion,
                                                                            max_generaciones=max_generaciones,
                                                                            mutacion=mutacion_opcion,
                                                                            recombinacion=recombinacion_opcion, 
                                                                        )

        valor_maximo_generacion.append(valor_maximo)
        valor_minimo_generacion.append(valor_minimo)
        valor_promedio_generacion.append(valor_promedio)
        valor_desviacion_std_generacion.append(valor_desviacion_std)
        valor_varianza_generacion.append(valor_varianza)
        valor_diversidad_generacion.append(valor_divercidad)

        
        poblacion = nueva_generacion


    grafica_promedio: Grafica = Grafica_lineas()
    grafica_maximo: Grafica = Grafica_lineas()
    grafica_minimo: Grafica = Grafica_lineas()
    grafica_desviacion_std: Grafica = Grafica_lineas()
    grafica_mejora_generacion: Grafica = Grafica_lineas()
    grafica_varianza: Grafica = Grafica_lineas()
    grafica_diversidad: Grafica = Grafica_lineas()

    grafica_promedio.generar_grafica(datos=valor_promedio_generacion,
                                     etiqueta="Promedio",
                                     guardar_grafica=("Promedio_"+configuracion_opcion))

    grafica_maximo.generar_grafica(datos=valor_maximo_generacion,
                                   etiqueta="Maximo",
                                   guardar_grafica=("Maximo_"+configuracion_opcion))

    grafica_minimo.generar_grafica(datos=valor_minimo_generacion,
                                   etiqueta="Minimo",
                                   guardar_grafica=("Minio_"+configuracion_opcion))

    grafica_desviacion_std.generar_grafica(datos=valor_desviacion_std_generacion,
                                           etiqueta="Desviación estándar",
                                           guardar_grafica=("DesviacionEstandar_"+configuracion_opcion))

    grafica_varianza.generar_grafica(datos=valor_varianza_generacion,
                                     etiqueta=" Varianza",
                                     guardar_grafica=("Varianza_"+configuracion_opcion))

    grafica_diversidad.generar_grafica(datos=valor_diversidad_generacion,
                                       etiqueta=" Diversidad",
                                       guardar_grafica=("Diversidad_"+configuracion_opcion))

    mejora = np.diff(valor_maximo_generacion)

    grafica_mejora_generacion.generar_grafica(datos=mejora,
                                              etiqueta="Mejora entre cada generación",
                                              guardar_grafica=("MejoraGeneraciones_"+configuracion_opcion))

# Opciones de calculo del fitnes para la penalización.
# - fitnes_cero
# - lineal
# - cuadratica
# - exponencial


# opciones para la Mutación y la recombinacion (Cruza)
# recombinacion (Cruce_cargas)
# - Cruce_uniforme
# - Cruce_un_punto
# - Cruce_dos_punto

# mutacion (Mutacion)
# - Seleccion_aleatoria
# - Tasa_mutacion


# --- Configuraciones ---
recombinacion_opcion: list[Cruce_cargas] = [Cruce_uniforme, Cruce_un_punto, Cruce_dos_punto]
mutacion_opcion: list[Mutacion]= [Seleccion_aleatoria,Tasa_mutacion]
penalizacion_opcion: list[str] = ['fitnes_cero','lineal','cuadratica','exponencial']
supervivencia_opcion: list[Supervivencia] = [Con_elitismo,Con_elitismo]

configuraciones_posibles:list[list[Cruce_cargas,Mutacion,str,Supervivencia]] = generar_combinaciones(recombinacion_opcion,mutacion_opcion,penalizacion_opcion,supervivencia_opcion)


evolucion()